##1. Setup e Configuração
Importação de bibliotecas essenciais e definição dinâmica do caminho dos dados no Repositório.

In [0]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

current_path = os.getcwd()
repo_name = "Grupo7-Setor-de-Seguros"

if repo_name in current_path:
    root_path = current_path.split(repo_name)[0] + repo_name
else:
    root_path = os.path.dirname(os.path.dirname(os.getcwd()))

caminho_arquivo = f"{root_path}/data/processed/prata/seguros_sinistros.csv"

print(f"📂 Diretório Raiz: {root_path}")
print(f"🎯 Arquivo Alvo: {caminho_arquivo}")

## 2. Leitura e Sanitização (Data Cleaning)
Leitura via Pandas (para contornar bloqueio de segurança) e conversão para Spark utilizando `try_cast`.
* **Objetivo:** Garantir que erros de formatação no CSV não quebrem a análise. Textos em colunas numéricas serão convertidos para `0.0`.

In [0]:
pdf = pd.read_csv(caminho_arquivo, sep=',', decimal='.', dtype=str, on_bad_lines='skip')
pdf = pdf.where(pd.notnull(pdf), None)

df_raw = spark.createDataFrame(pdf)

cols_numericas = ["valor_premio", "valor_pagamento", "capital_segurado", "valor_sinistro"]

df_raw = df_raw
for col_name in cols_numericas:
    if col_name in df_raw.columns:
        df_raw = df_raw.withColumn(
            col_name,
            F.coalesce(F.col(col_name).cast(DoubleType()), F.lit(0.0))
        )

for col_text in ["nome_contratante", "nome_beneficiario", "tipo_sinistro"]:
    df_raw = df_raw.withColumn(col_text, F.trim(F.col(col_text)))

df_sinistros = df_raw.filter(F.col("tipo_sinistro").isNotNull()) \
    .dropDuplicates(["nome_contratante", "data_contratacao", "valor_sinistro", "tipo_sinistro"])

df_seguros = df_raw.dropDuplicates(["nome_contratante", "valor_premio", "capital_segurado"])

print("Dados carregados!")


## 3. Engenharia de Atributos
* **REGIAO:** Agrupamento de estados.
* **SEXO:** Classificação via Primeiro Nome (IBGE + Heurística).
* **QUARTIS:** Segmentação de Capital.

In [0]:
import pyspark.sql.functions as F

df_prep = df_seguros.withColumn("REGIAO", 
    F.when(F.col("estado_contratante").isin("SP","RJ","MG","ES"), "Sudeste")
    .when(F.col("estado_contratante").isin("PR","SC","RS"), "Sul")
    .when(F.col("estado_contratante").isin("PE","BA","CE","MA","PB","AL","SE","RN","PI"), "Nordeste")
    .when(F.col("estado_contratante").isin("AM","PA","AC","RR","RO","TO","AP"), "Norte")
    .otherwise("Centro-Oeste")
)

df_prep = df_prep.withColumn("primeiro_nome", F.split(F.col("nome_contratante"), " ").getItem(0))

nomes_fem_excecao = ["Julie","Alice","Beatriz","Ines","Raquel","Liz","Isabel"]

df_prep = df_prep.withColumn(
    "SEXO",
    F.when(F.col("primeiro_nome").isin(nomes_fem_excecao), "Feminino")
    .when(F.lower(F.col("primeiro_nome")).endswith("a"), "Feminino")
    .otherwise("Masculino")
).drop("primeiro_nome")

q1, q3 = df_prep.approxQuantile("capital_segurado", [0.25, 0.75], 0.01)

df_prep = df_prep.withColumn("ACIMA_Q3_CAPITAL", F.col("capital_segurado") > q3)
df_prep = df_prep.withColumn("ABAIXO_Q1_CAPITAL", F.col("capital_segurado") < q1)

df_prep.createOrReplaceTempView("prep")

display(df_prep.limit(5))


In [0]:
import pyspark.sql.functions as F

df_freq_acidentes = df_sinistros.groupBy("nome_contratante") \
    .agg(F.count("tipo_sinistro").alias("qtd_acidentes"))

df_freq_contratos = df_prep.groupBy("nome_contratante") \
    .agg(F.count("nome_contratante").alias("qtd_contratos"))
df_analise = df_prep.join(df_freq_acidentes, on="nome_contratante", how="left") \
                    .join(df_freq_contratos, on="nome_contratante", how="left") \
                    .fillna(0, subset=["qtd_acidentes"])

df_analise = df_analise.withColumn(
    "razao_pag_cap", 
    F.when(F.col("capital_segurado") > 0, F.col("valor_pagamento") / F.col("capital_segurado"))
     .otherwise(0)
)

df_analise = df_analise.withColumn(
    "razao_contrato_acidente",
    F.when(F.col("qtd_acidentes") > 0, F.col("qtd_contratos") / F.col("qtd_acidentes"))
     .otherwise(F.col("qtd_contratos"))
)

df_analise.createOrReplaceTempView("analise_completa")
print("✅ Dataset Mestre criado com sucesso! Variável: df_analise")
display(df_analise.limit(5))

## Task A: Análise de Autosseguro
Verificação de casos onde Contratante e Beneficiário são a mesma pessoa.
> **Nota:** Se o resultado for 0, indica uma regra de negócio da base onde os papéis são excludentes por contrato.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import quarter

df_auto = (
    df_prep
    .filter(F.upper("nome_contratante") == F.upper("nome_beneficiario"))
    .withColumn("TRIMESTRE", quarter("data_contratacao"))
)

count_auto = df_auto.count()
print(f"Registros de autosseguro: {count_auto}")

if count_auto == 0:
    print("⚠️ Nenhum autosseguro encontrado.")


In [0]:
if count_auto == 0:
    print("⚠️ Nenhum autosseguro encontrado. Análises interrompidas.")
else:
    print("🔎 Executando análises da Task A…")

    print("\n=== (A1) CORRELAÇÕES POR TRIMESTRE E SEXO ===")

    segmentos = (
        df_auto
        .select("TRIMESTRE", "SEXO")
        .dropna()
        .distinct()
        .collect()
    )

    for seg in segmentos:
        trimestre = seg["TRIMESTRE"]
        sexo = seg["SEXO"]

        df_seg = df_auto.filter(
            (F.col("TRIMESTRE") == trimestre) &
            (F.col("SEXO") == sexo)
        )

        if df_seg.count() > 1:
            try:
                corr_premio_cap = df_seg.stat.corr("valor_premio", "capital_segurado")
                corr_pag_premio = df_seg.stat.corr("valor_pagamento", "valor_premio")

                print(f"[Sexo={sexo} | T{trimestre}] "
                      f"Corr(Prêmio x Capital) = {corr_premio_cap:.4f} | "
                      f"Corr(Pagamento x Prêmio) = {corr_pag_premio:.4f}")

            except:
                print(f"[Sexo={sexo} | T{trimestre}] Dados insuficientes.")

    print("\n=== (A2) ESTATÍSTICAS DA RAZÃO PRÊMIO/PAGAMENTO ===")

    df_stats = df_auto.withColumn(
        "razao_premio_pgt",
        F.when(F.col("valor_pagamento") > 0,
               F.col("valor_premio") / F.col("valor_pagamento"))
    )

    display(
        df_stats.select("razao_premio_pgt")
        .summary("mean", "50%", "stddev", "min", "max")
    )

    print("\n=== (A3) FREQUÊNCIA POR REGIÃO ===")

    display(
        df_auto.groupBy("REGIAO")
        .count()
        .orderBy(F.col("count").desc())
    )

    print("\n=== (A4) SEXO E QUANTIDADE DE ACIDENTES ===")

    df_freq = (
        df_sinistros.groupBy("nome_contratante")
        .agg(F.count("tipo_sinistro").alias("qtd_acidentes"))
    )

    df_join = (
        df_auto.join(df_freq, "nome_contratante", "left")
        .fillna(0, ["qtd_acidentes"])
    )

    display(
        df_join.select("nome_contratante", "SEXO", "REGIAO", "qtd_acidentes")
    )


## Task B: Clientes Mais Frequentes (Heavy Users)
Identificação do perfil de risco (Top 10% em frequência de acidentes).

In [0]:
print("=== TASK B: Análise de Clientes Mais e Menos Frequentes ===")

df_mais_frequentes = df_analise.orderBy(F.col("qtd_contratos").desc()).limit(10)
df_menos_frequentes = df_analise.orderBy(F.col("qtd_contratos").asc()).limit(10)

print(">>> Perfil dos Clientes MAIS Frequentes (Top 20 com mais contratos)")
display(df_mais_frequentes.select(
    "nome_contratante", "REGIAO", "SEXO", "qtd_acidentes", "razao_pag_cap"
))

print(">>> Perfil dos Clientes MENOS Frequentes (Top 20 com menos contratos)")
display(df_menos_frequentes.select(
    "nome_contratante", "REGIAO", "SEXO", "qtd_acidentes", "razao_pag_cap"
))

## Task C: Análise por Faixa de Capital
Comparativo demográfico:
* **Alto Capital:** Acima do 3º Quartil.
* **Baixo Capital:** Abaixo do 1º Quartil.

In [0]:
print("=== TASK C: Análise de Extremos de Capital (Q1 e Q3) ===")

df_grupos_capital = df_analise.withColumn(
    "grupo_capital",
    F.when(F.col("ACIMA_Q3_CAPITAL") == True, "Alto Capital (>Q3)")
     .when(F.col("ABAIXO_Q1_CAPITAL") == True, "Baixo Capital (<Q1)")
     .otherwise("Médio Capital")
)

display(
    df_grupos_capital.groupBy("grupo_capital", "REGIAO", "SEXO")
    .agg(
        F.count("nome_contratante").alias("total_clientes"),
        F.avg("qtd_acidentes").alias("media_acidentes"),
        F.max("qtd_acidentes").alias("max_acidentes"),
        F.avg("valor_premio").alias("media_premio")
    )
    .orderBy("grupo_capital", "REGIAO")
)

## Task D: Eficiência da Carteira (Contratos por Acidente)
Métrica de rentabilidade/risco.
* **Valor Alto:** Região eficiente (muitos contratos para poucos acidentes).
* **Valor Baixo:** Região de risco (alta sinistralidade proporcional).

In [0]:
print("=== TASK D: Razão Contratos / Acidentes ===")

def get_percentiles(col_name):
    return [
        F.expr(f"percentile_approx({col_name}, 0.5)").alias("mediana"),
        F.expr(f"percentile_approx({col_name}, 0.25)").alias("Q1"),
        F.expr(f"percentile_approx({col_name}, 0.75)").alias("Q3")
    ]

print(">>> Estatísticas GERAIS da Razão Contratos/Acidentes")
display(df_analise.select("razao_contrato_acidente").summary())

print(">>> Razão por SEXO (Média, Mediana, Q1, Q3)")
display(
    df_analise.groupBy("SEXO")
    .agg(
        F.avg("razao_contrato_acidente").alias("media"),
        *get_percentiles("razao_contrato_acidente")
    )
)

print(">>> Razão por REGIÃO")
display(
    df_analise.groupBy("REGIAO")
    .agg(
        F.avg("razao_contrato_acidente").alias("media"),
        *get_percentiles("razao_contrato_acidente")
    )
)

print(">>> Razão por GRUPOS DE CAPITAL")
df_grupos_capital_d = df_analise.withColumn(
    "grupo_capital",
    F.when(F.col("ACIMA_Q3_CAPITAL") == True, "Alto Capital (>Q3)")
     .when(F.col("ABAIXO_Q1_CAPITAL") == True, "Baixo Capital (<Q1)")
     .otherwise("Médio Capital")
)

display(
    df_grupos_capital_d.groupBy("grupo_capital")
    .agg(
        F.avg("razao_contrato_acidente").alias("media"),
        *get_percentiles("razao_contrato_acidente")
    )
)